## LightGBM
### O LightGBM é um framework de gradient boosting ideal para conjuntos de dados pequenos. O mesmo utiliza algoritmos de aprendizado baseados em arvores

#### Cada resposta leva a outro caminho até chegar a uma previsão. O LightGBM constrói várias árvores em sequência. A cada etapa, a nova árvore tenta melhorar as previsões feitas pelo conjunto anterior. Essa técnica se chama gradient boosting.

### Primeiro algoritmo: 
#### Gerar Acurácia, Matriz de Confusão e F1 Score do dataset de treino

In [11]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [1]:
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Recarregar o arquivo original
df_modelo = pd.read_csv(
    "csv_malware/IIoT_Malware_Timeseries_Dataset_modificado.csv",
    low_memory=False
)

print("Dataset original:", df_modelo.shape)
print(df_modelo["tipo_conexao"].value_counts(dropna=False))

# 2. Selecionar variáveis numéricas
X = (
    df_modelo
    .select_dtypes(include="number")
    .drop(
        columns=["Source IP", "Destination IP"],
        errors="ignore"
    )
    .copy()
)

# 3. Converter a classe
y = (
    df_modelo["tipo_conexao"]
    .astype(str)
    .str.strip()
    .map({
        "Benign": 0,
        "Malign": 1
    })
)

print("Formato de X:", X.shape)
print("Classes:")
print(y.value_counts(dropna=False))

# Verificação de segurança
if y.isna().any():
    print("Valores não reconhecidos:")
    print(df_modelo.loc[y.isna(), "tipo_conexao"].value_counts(dropna=False))
    raise ValueError("Existem classes que não foram convertidas.")

# 4. Dividir os dados
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 5. Treinar o LightGBM
modelo = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    class_weight="balanced",
    verbosity=-1,
)

modelo.fit(X_treino, y_treino)

# 6. Avaliar
y_previsto = modelo.predict(X_teste)

print("Acurácia:", accuracy_score(y_teste, y_previsto))
print("\nMatriz de confusão:")
print(confusion_matrix(y_teste, y_previsto))

print("\nRelatório:")
print(
    classification_report(
        y_teste,
        y_previsto,
        target_names=["Benign", "Malign"]
    )
)

Dataset original: (45289, 47)
tipo_conexao
Benign    31621
Malign    13668
Name: count, dtype: int64
Formato de X: (45289, 39)
Classes:
tipo_conexao
0    31621
1    13668
Name: count, dtype: int64
Acurácia: 0.5485758445572975

Matriz de confusão:
[[3902 2422]
 [1667 1067]]

Relatório:
              precision    recall  f1-score   support

      Benign       0.70      0.62      0.66      6324
      Malign       0.31      0.39      0.34      2734

    accuracy                           0.55      9058
   macro avg       0.50      0.50      0.50      9058
weighted avg       0.58      0.55      0.56      9058

